In [1]:
import os
from pathlib import Path
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import QuantizationConfig

# Для валидации
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification

W1126 14:13:51.973000 24224 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
#модель "Специалист"
model_checkpoint_path = Path("../saved_model_specific_best/") 
onnx_path = Path("onnx_models/specific/") # Папка, куда сохранится сжатая модель

onnx_path.mkdir(parents=True, exist_ok=True)

print(f"Исходная модель: {model_checkpoint_path}")
print(f"Целевая папка ONNX: {onnx_path}")

Исходная модель: ..\saved_model_specific_best
Целевая папка ONNX: onnx_models\specific


In [8]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig

# конвертация (export=True)
ort_model = ORTModelForSequenceClassification.from_pretrained(
    model_checkpoint_path, 
    export=True
)

quantizer = ORTQuantizer.from_pretrained(ort_model)

# avx2() создает идеальный конфиг для динамической квантизации на CPU (Intel/AMD)
qconfig = AutoQuantizationConfig.avx2(
    is_static=False,    # Динамическая квантизация
    per_channel=False 
)

qconfig.operators_to_quantize = ['MatMul', 'Add']

quantizer.quantize(
    save_dir=onnx_path,
    quantization_config=qconfig,
)

print(f"Готово! Модель успешно квантизирована и сохранена в: {onnx_path}")

Готово! Модель успешно квантизирована и сохранена в: onnx_models\specific


In [10]:
original_size = os.path.getsize(model_checkpoint_path / "model.safetensors") / (1024 * 1024)
quantized_size = os.path.getsize(onnx_path / "model_quantized.onnx") / (1024 * 1024)

print(f"Размер оригинальной модели (PyTorch): {original_size:.2f} MB")
print(f"Размер квантизированной модели (ONNX int8): {quantized_size:.2f} MB")
print(f"Сжатие в {original_size / quantized_size:.2f} раз(а)")

Размер оригинальной модели (PyTorch): 44.98 MB
Размер квантизированной модели (ONNX int8): 38.25 MB
Сжатие в 1.18 раз(а)


In [11]:
# Загрузка данных для валидации
# Убедитесь, что пути к файлам верные

try:
    df_specific = pd.read_csv('../data/TRAINING_DATASET_SPECIFIC.csv')
    X = df_specific['processed_text'].fillna('')
    y = df_specific['label']
    
    #тот же самый энкодер, что и при обучении!
    label_encoder_specific = joblib.load('../label_encoder_specific.joblib')
    y_encoded = label_encoder_specific.transform(y)

    # те же самые тестовые данные (random_state=42 гарантирует тот же сплит)
    _, X_test, _, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
    print("Данные успешно загружены и разделены.")
    
except FileNotFoundError as e:
    print(f"Ошибка: Не найден файл данных или энкодер. Проверьте пути. {e}")

Данные успешно загружены и разделены.


In [12]:
# Модель загружается из папки onnx_path с помощью ORTModelForSequenceClassification
quantized_model = ORTModelForSequenceClassification.from_pretrained(onnx_path)

# токенизатор из оригинальной папки (он не меняется при квантизации)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_path)

print("ONNX модель и токенизатор загружены.")

Could not find any ONNX files with standard file name model.onnx, files found: [WindowsPath('model_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


ONNX модель и токенизатор загружены.


In [13]:
# Запуск предсказаний
predictions = []

print("Запуск инференса на квантизированной модели...")
for text in tqdm(X_test.tolist()):
    # Токенизация
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # метод вызова такой же, как в PyTorch
    outputs = quantized_model(**inputs)
    
    # Получение класса (argmax)
    # output.logits в ONNX Runtime возвращает numpy array, detach() не нужен
    logits = outputs.logits
    if hasattr(logits, 'detach'): # На случай если вернется torch tensor
        logits = logits.detach().numpy()
        
    predicted_class_id = np.argmax(logits)
    predictions.append(predicted_class_id)

new_f1 = f1_score(y_test, predictions, average='macro')

# Сравнение результатов
original_f1 = 0.999 

print("\n" + "="*30)
print(f"F1-score ОРИГИНАЛЬНОЙ модели: {original_f1}")
print(f"F1-score КВАНТИЗИРОВАННОЙ модели: {new_f1:.4f}")

# Проверяем падение
f1_drop = (original_f1 - new_f1) / original_f1 * 100
print(f"Падение качества: {f1_drop:.2f}%")

if f1_drop > 5:
    print("ВНИМАНИЕ: Падение качества превышает 5%!")
else:
    print("Отлично! Падение качества в пределах нормы.")
print("="*30)

Запуск инференса на квантизированной модели...


100%|███████████████████████████████████████████████████████████████████████████████████████████| 5329/5329 [00:07<00:00, 688.97it/s]


F1-score ОРИГИНАЛЬНОЙ модели: 0.999
F1-score КВАНТИЗИРОВАННОЙ модели: 0.9992
Падение качества: -0.02%
Отлично! Падение качества в пределах нормы.


In [14]:
model_checkpoint_path_gen = Path("../saved_model_general_best/") 
onnx_path_gen = Path("onnx_models/general/") 
onnx_path_gen.mkdir(parents=True, exist_ok=True)

print(f"--- Обработка модели GENERAL ---")
print(f"Исходная: {model_checkpoint_path_gen}")
print(f"Целевая: {onnx_path_gen}")

# Экспорт в ONNX
ort_model_gen = ORTModelForSequenceClassification.from_pretrained(
    model_checkpoint_path_gen, 
    export=True
)

quantizer_gen = ORTQuantizer.from_pretrained(ort_model_gen)
qconfig_gen = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
qconfig_gen.operators_to_quantize = ['MatMul', 'Add']

quantizer_gen.quantize(
    save_dir=onnx_path_gen,
    quantization_config=qconfig_gen,
)
print("Модель General успешно квантизирована!")

# Проверка размера
size_orig = os.path.getsize(model_checkpoint_path_gen / "model.safetensors") / (1024 * 1024)
size_quant = os.path.getsize(onnx_path_gen / "model_quantized.onnx") / (1024 * 1024)
print(f"Размер Original: {size_orig:.2f} MB")
print(f"Размер Quantized: {size_quant:.2f} MB")

# Проверка качества (Валидация)
try:
    df_gen = pd.read_csv('../data/TRAINING_DATASET_GENERAL.csv')
    X_gen = df_gen['processed_text'].fillna('')
    y_gen = df_gen['label']
    
    label_encoder_gen = joblib.load('../label_encoder_general.joblib') 
    y_encoded_gen = label_encoder_gen.transform(y_gen)

    _, X_test_gen, _, y_test_gen = train_test_split(
        X_gen, y_encoded_gen, test_size=0.2, random_state=42, stratify=y_encoded_gen
    )
    
    model_gen_quant = ORTModelForSequenceClassification.from_pretrained(onnx_path_gen)
    tokenizer_gen = AutoTokenizer.from_pretrained(model_checkpoint_path_gen)
    
    print("Начинаем тест качества General...")
    preds_gen = []
    for text in tqdm(X_test_gen.tolist()):
        inputs = tokenizer_gen(text, return_tensors="pt", truncation=True, padding=True)
        outputs = model_gen_quant(**inputs)
        # Получаем логиты и argmax
        logits = outputs.logits
        if hasattr(logits, 'detach'): logits = logits.detach().numpy()
        preds_gen.append(np.argmax(logits))

    f1_gen = f1_score(y_test_gen, preds_gen, average='macro')
    print(f"F1-score General (Quantized): {f1_gen:.4f}")

except Exception as e:
    print(f"Ошибка при валидации General: {e}")

--- Обработка модели GENERAL ---
Исходная: ..\saved_model_general_best
Целевая: onnx_models\general


Could not find any ONNX files with standard file name model.onnx, files found: [WindowsPath('model_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Модель General успешно квантизирована!
Размер Original: 44.96 MB
Размер Quantized: 38.25 MB
Начинаем тест качества General...


100%|███████████████████████████████████████████████████████████████████████████████████████████| 3841/3841 [00:06<00:00, 618.33it/s]

F1-score General (Quantized): 0.8480
